# 📥 Fase 2: Ingestão e Processamento de Dados Agrícolas (Notebook de Estudo)

Este notebook detalha o processo de **limpeza, sanitização, pivotagem e consolidação** dos dados brutos obtidos da API SIDRA do IBGE (Tabela 5457). 

### 🎯 Objetivos de Aprendizado:
1. **Entender a Estrutura SIDRA**: Lidar com metadados e o formato longo retornado pela API.
2. **Limpeza e Sanitização**: Converter os caracteres especiais do IBGE (`-`, `..`, `...`, `X`) para representações numéricas adequadas no Pandas.
3. **Dominar a Pivotagem**: Transformar dados do formato vertical (longo) para colunas horizontais (largo), essencial para algoritmos de aprendizado de máquina.
4. **Consolidação**: Unificar dados de soja, milho e trigo em uma única base de dados otimizada e salva em Parquet.

## ⚙️ Configuração do Ambiente e Imports

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao sys.path para imports absolutos
BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

In [ ]:
import json
import numpy as np
import pandas as pd
from src.ingestion.sidra_mapping import (
    SidraCrops,
    SidraVariables,
    SIDRA_RENAME_MAP,
)

print("Mapeamento primário de colunas (SIDRA_RENAME_MAP):\n", json.dumps(SIDRA_RENAME_MAP, indent=4))

## 📖 Passo 1: Leitura de Dados Brutos e Descarte de Metadados

A API do SIDRA sempre retorna os metadados explicativos das colunas na primeira linha (índice 0) do JSON. Demonstramos o descarte dessa linha usando `.iloc[1:]`.

In [ ]:
sample_raw_path = BASE_DIR / "data" / "raw" / "pam_parana_soja_raw.json"

# Carrega com o Pandas
df_raw = pd.read_json(sample_raw_path)
print("--- Primeira Linha (Metadados Explicativos) ---")
print(df_raw.iloc[0].to_dict())

# Descarte dos metadados
df_clean = df_raw.iloc[1:].copy()
print(f"\nTotal de registros lidos: {len(df_raw)} | Pós-limpeza de cabeçalho: {len(df_clean)}")
df_clean.head(2)

## 📋 Passo 2: Filtragem de Colunas e Renomeação Primária

Mantemos apenas as colunas que possuem dados de interesse (Ano, Município, Variável e Valor) para simplificar a manipulação.

In [ ]:
# Seleciona as colunas de acordo com as chaves do map
df_filtered = df_clean[list(SIDRA_RENAME_MAP.keys())].rename(columns=SIDRA_RENAME_MAP)
df_filtered.head()

## 🛡️ Passo 3: Sanitização de Símbolos Especiais do IBGE

O IBGE adota caracteres para designar situações específicas de dados (ex: `-` para zero absoluto e `X` para segredo estatístico). 
Se tentarmos converter a coluna `valor` diretamente para numérico, o Pandas gerará erro ou anomalias. Aplicamos a regra de sanitização regulamentada no projeto:

In [ ]:
print("Valores não numéricos únicos antes da limpeza:")
non_numeric = df_filtered[~df_filtered["valor"].astype(str).str.replace(".", "", 1).str.isdigit()]["valor"].unique()
print(non_numeric)

# Aplica o mapeamento de sanitização
df_filtered["valor"] = df_filtered["valor"].replace({
    "-": "0",
    "..": np.nan,
    "...": np.nan,
    "X": np.nan,
    "x": np.nan,
})

# Conversão estrita de tipos
df_filtered["valor"] = pd.to_numeric(df_filtered["valor"], errors="coerce")
df_filtered["ano"] = pd.to_numeric(df_filtered["ano"], errors="coerce").astype("Int64")
df_filtered["municipio_codigo"] = pd.to_numeric(df_filtered["municipio_codigo"], errors="coerce").astype("Int64")

print("\nTipagem final das colunas:")
df_filtered.info()

## 🔄 Passo 4: A Importância da Pivotagem (Longo para Largo)

Os dados do SIDRA são retornados no **Formato Longo**, onde cada variável de um município/ano ocupa uma linha individual. 
Para treinar modelos e calcular métricas entre colunas, precisamos convertê-los para o **Formato Largo**.

In [ ]:
print("--- Exemplo do Formato Longo Original (Filtro para Abatiá em 2010) ---")
print(df_filtered[df_filtered["municipio_nome"].str.contains("Abatiá") & (df_filtered["ano"] == 2010)][["municipio_nome", "ano", "variavel_codigo", "valor"]])

# Aplica o pivot_table
df_pivot = df_filtered.pivot_table(
    index=["municipio_codigo", "municipio_nome", "ano"],
    columns="variavel_codigo",
    values="valor",
    aggfunc="first"
).reset_index()

print("\n--- Exemplo do Formato Largo Pós-Pivotagem (Abatiá em 2010) ---")
df_pivot[df_pivot["municipio_nome"].str.contains("Abatiá") & (df_pivot["ano"] == 2010)]

## 🏷️ Passo 5: Renomeação das Colunas Baseada no Enum de Variáveis

O pivot gera colunas nomeadas com os IDs da API (`112`, `214`, etc.). Vamos traduzi-las usando a classe `SidraVariables` para nomes amigáveis em minúsculo.

In [ ]:
variables_mapping = {v.value: v.name.lower() for v in SidraVariables}
print("Mapeamento de Tradução de Colunas:\n", json.dumps(variables_mapping, indent=4))

df_final = df_pivot.rename(columns=variables_mapping)
df_final.head()

## 🚀 Passo 6: Consolidação Global (Soja, Milho e Trigo)

Agora consolidamos o processo completo para as três culturas especificadas no projeto, adicionando a coluna `produto` para diferenciá-las, concatenando os resultados e salvando em Parquet.

In [ ]:
locality_name = "parana"
raw_dir = BASE_DIR / "data" / "raw"
dfs_all = []

for crop in SidraCrops:
    crop_name = crop.name.lower()
    file_path = raw_dir / f"pam_{locality_name}_{crop_name}_raw.json"
    
    print(f"Processando base de {crop_name.upper()} de: {file_path.name}")
    
    # 1. Carrega dados
    df = pd.read_json(file_path).iloc[1:]
    
    # 2. Renomeia e filtra
    df_2 = df[list(SIDRA_RENAME_MAP.keys())].rename(columns=SIDRA_RENAME_MAP)
    
    # 3. Sanitiza
    df_2["valor"] = df_2["valor"].replace({
        "-": "0",
        "..": np.nan,
        "...": np.nan,
        "X": np.nan,
        "x": np.nan,
    })
    df_2["valor"] = pd.to_numeric(df_2["valor"], errors="coerce")
    df_2["ano"] = pd.to_numeric(df_2["ano"], errors="coerce").astype("Int64")
    df_2["municipio_codigo"] = pd.to_numeric(df_2["municipio_codigo"], errors="coerce").astype("Int64")
    
    # 4. Pivotagem
    df_pivot = df_2.pivot_table(
        index=["municipio_codigo", "municipio_nome", "ano"],
        columns="variavel_codigo",
        values="valor",
        aggfunc="first"
    ).reset_index()
    
    # 5. Mapeamento final de colunas e identificador de produto
    df_crop_final = df_pivot.rename(columns=variables_mapping)
    df_crop_final["produto"] = crop_name
    
    dfs_all.append(df_crop_final)

# Concatena tudo
df_consolidated = pd.concat(dfs_all, ignore_index=True)
print(f"\nConsolidação finalizada. Total de linhas geradas: {len(df_consolidated)}")

# Salva no disco no formato parquet de alta performance
output_parquet = BASE_DIR / "data" / "processed" / "pam_parana_consolidado_study.parquet"
df_consolidated.to_parquet(output_parquet, index=False, engine="pyarrow")
print(f"Arquivo de estudo salvo em: {output_parquet}")

## 📊 Passo 7: Análise Exploratória Rápida

Podemos verificar a integridade da nossa base consolidada com métricas rápidas.

In [ ]:
# Médias gerais por produto ao longo de toda a série histórica
print("Médias históricas por cultura no Paraná (2010-2024):")
df_consolidated.groupby("produto")[["quantidade_produzida", "area_plantada", "rendimento_medio"]].mean().round(2)